<div style="display: flex; justify-content: flex-start; align-items: center;">
   <a href="https://colab.research.google.com/github/msfasha/307307-BI-Methods-Generative-AI/blob/main/20252/Module%204%20-%20Testing%20the%20Transformers%20Library/2-Extended_Pipelines-Python.ipynb" target="_parent"><img 
   src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
</div>

# Extended Hugging Face Pipelines

This notebook extends the pipelines introduced in the previous notebook by demonstrating additional capabilities available through the Hugging Face `pipeline()` API.

Each section uses a different task type to show the breadth of what a single API call can accomplish:

| # | Pipeline | Task Type |
|---|----------|-----------|
| 1 | Text-to-Speech | Audio generation |
| 2 | Speech-to-Text (ASR) | Audio understanding |
| 3 | Zero-Shot Classification | Text classification without training |
| 4 | Image Captioning | Vision → Text |
| 5 | Visual Question Answering | Vision + Text → Text |
| 6 | Table Question Answering | Structured data → Text |

#### Setup — Install Required Libraries

In [ ]:
%pip install -q transformers torch torchaudio sentencepiece sacremoses Pillow requests scipy datasets

---
## 1. Text-to-Speech (TTS)

Text-to-Speech converts written text into spoken audio. The model used here is `suno/bark-small`, a lightweight version of Bark — a generative audio model that can produce natural-sounding speech.

> **Note:** This pipeline downloads a model (~900 MB) and is slow on CPU. Using a GPU (e.g., Colab T4) is recommended.

In [ ]:
from transformers import pipeline
import scipy
from IPython.display import Audio

tts = pipeline("text-to-speech", model="suno/bark-small")

text = "Welcome to the Generative AI course. Today we explore text-to-speech synthesis."

result = tts(text)

# Save to file
scipy.io.wavfile.write("tts_output.wav", rate=result["sampling_rate"], data=result["audio"].squeeze())
print("Audio saved to tts_output.wav")

# Play inline (works in Colab and Jupyter)
Audio(result["audio"].squeeze(), rate=result["sampling_rate"])

---
## 2. Automatic Speech Recognition (ASR)

ASR transcribes spoken audio into text. We use OpenAI's **Whisper** model, available through the Hugging Face pipeline.

Whisper supports multiple languages including Arabic. Here we transcribe the audio file we generated in the previous section.

In [ ]:
from transformers import pipeline

asr = pipeline("automatic-speech-recognition", model="openai/whisper-small")

# Transcribe the audio file generated in Section 1
result = asr("tts_output.wav")

print("Transcription:", result["text"])

---
## 3. Zero-Shot Classification

Zero-shot classification assigns a text to one of your custom labels **without any training**. The model reasons about which label best matches the text using its pre-trained language understanding.

This is extremely useful in BI scenarios where labels change frequently or labelled training data is unavailable.

In [ ]:
from transformers import pipeline

classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# Example 1: Customer support ticket routing
ticket = "My invoice shows a charge I don't recognize. I need this investigated immediately."
labels = ["billing issue", "technical support", "account access", "product question"]

result = classifier(ticket, candidate_labels=labels)

print("Ticket:", ticket)
print()
for label, score in zip(result["labels"], result["scores"]):
    print(f"  {label:<25} {score:.4f}")

In [ ]:
# Example 2: Classify news headlines into business topics
headlines = [
    "Central bank raises interest rates for the third consecutive quarter.",
    "New AI regulation proposed by the European Commission.",
    "Oil prices surge following supply chain disruptions.",
    "Startup raises $50M in Series B funding round."
]

topics = ["finance", "technology", "energy", "entrepreneurship", "politics"]

for headline in headlines:
    result = classifier(headline, candidate_labels=topics)
    top_label = result["labels"][0]
    top_score = result["scores"][0]
    print(f"Headline : {headline}")
    print(f"Category : {top_label} ({top_score:.4f})")
    print()

---
## 4. Image Captioning

Image captioning generates a natural language description of an image. This is a **multimodal** pipeline — the model processes both visual and textual information.

We use **BLIP** (Bootstrapped Language-Image Pre-training) from Salesforce.

In [ ]:
from transformers import pipeline
from PIL import Image
import requests
from IPython.display import display

captioner = pipeline("image-to-text", model="Salesforce/blip-image-captioning-base")

# Load a sample image from the web
url = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/1200px-Cat03.jpg"
image = Image.open(requests.get(url, stream=True).raw).convert("RGB")

display(image)

result = captioner(image)
print("Caption:", result[0]["generated_text"])

In [ ]:
# Caption multiple images
image_urls = [
    "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/1200px-Cat03.jpg",
    "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png",
    "https://upload.wikimedia.org/wikipedia/commons/thumb/a/a7/Camponotus_flavomarginatus_ant.jpg/640px-Camponotus_flavomarginatus_ant.jpg"
]

for url in image_urls:
    img = Image.open(requests.get(url, stream=True).raw).convert("RGB")
    caption = captioner(img)[0]["generated_text"]
    print(f"URL   : {url.split('/')[-1]}")
    print(f"Caption: {caption}")
    print()

---
## 5. Visual Question Answering (VQA)

VQA takes an image **and** a question as input, then answers the question based on the visual content. This extends image captioning to interactive, targeted queries — for example, asking questions about a business chart or dashboard screenshot.

In [ ]:
from transformers import pipeline
from PIL import Image
import requests
from IPython.display import display

vqa = pipeline("visual-question-answering", model="Salesforce/blip-vqa-base")

# Load a sample image
url = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/1200px-Cat03.jpg"
image = Image.open(requests.get(url, stream=True).raw).convert("RGB")

display(image)

# Ask questions about the image
questions = [
    "What animal is in the image?",
    "What color is the animal?",
    "Is the animal indoors or outdoors?"
]

for question in questions:
    result = vqa(image=image, question=question)
    print(f"Q: {question}")
    print(f"A: {result[0]['answer']} (score: {result[0]['score']:.4f})")
    print()

---
## 6. Table Question Answering

Table QA answers natural language questions over structured tabular data. The model used is **TAPAS** (Table Parser), developed by Google.

This pipeline is highly relevant to **Business Intelligence** — it lets users query data tables using plain language without writing SQL or code.

In [ ]:
from transformers import pipeline
import pandas as pd

tqa = pipeline("table-question-answering", model="google/tapas-base-finetuned-wtq")

# Sample sales data table
data = {
    "Product": ["Laptop", "Tablet", "Smartphone", "Monitor", "Keyboard"],
    "Category": ["Electronics", "Electronics", "Electronics", "Electronics", "Accessories"],
    "Units Sold": ["120", "340", "890", "210", "650"],
    "Revenue (USD)": ["144000", "136000", "534000", "84000", "52000"],
    "Region": ["North", "South", "North", "East", "West"]
}

table = pd.DataFrame(data)
print(table.to_string(index=False))
print()

In [ ]:
# Ask natural language questions over the table
questions = [
    "Which product had the highest revenue?",
    "How many units of Keyboard were sold?",
    "What is the total revenue?",
    "Which products are in the North region?"
]

for question in questions:
    result = tqa(table=table, query=question)
    print(f"Q: {question}")
    print(f"A: {result['answer']}")
    print()

---
## Summary

All six pipelines above use the same `pipeline()` interface from Hugging Face Transformers:

```python
pipe = pipeline(task, model=model_name)
result = pipe(input)
```

| Pipeline | Model Used |
|----------|------------|
| Text-to-Speech | `suno/bark-small` |
| Speech-to-Text | `openai/whisper-small` |
| Zero-Shot Classification | `facebook/bart-large-mnli` |
| Image Captioning | `Salesforce/blip-image-captioning-base` |
| Visual Question Answering | `Salesforce/blip-vqa-base` |
| Table Question Answering | `google/tapas-base-finetuned-wtq` |

The Hugging Face Hub hosts thousands of models — you can swap any model name to try different architectures on the same task.